# NORT Inference — Apply Trained Classifier to NOVEL Videos

This notebook applies a trained SVM classifier to all NOVEL-phase NORT videos to produce per-frame exploration predictions.

**Inputs:**
- `svm_exploratory_classifier.pkl` — trained RBF-SVM
- `feature_scaler.pkl` — fitted StandardScaler from training
- NOVEL-phase `.mp4` videos + their DLC `.csv` files
- Per-video `_bbox_assignments.pkl` files (bounding box positions for each frame)

**Pipeline overview:**
1. Load the trained model and scaler
2. For each NOVEL video, extract the same geometric features used during training
3. Predict exploratory / non-exploratory for each frame
4. If exploratory, assign the exploration to the closest object by nose-to-centroid distance
5. Save per-video JSON prediction files and labeled MP4s with overlaid annotations

**Output:** Per-video `_predictions.json` files aggregated into `nort_results.csv` for use in `Correlate-MoSeq-And-Traditional-Behavior.ipynb`

In [1]:
import json
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
import os
import cv2
import shutil
from tqdm import tqdm
import joblib
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# Load the trained model and scaler
model_dir = "/mnt/g/Shared drives/llorente-lab/Moseq/AnalyzedData/Revision-NORT/nort_model"
model_file = os.path.join(model_dir, "svm_exploratory_classifier.pkl")
scaler_file = os.path.join(model_dir, "feature_scaler.pkl")

svm_optimized = joblib.load(model_file)
scaler = joblib.load(scaler_file)

print("Model and scaler loaded!")
print(f"Model: {type(svm_optimized)}")
print(f"Scaler: {type(scaler)}")

Model and scaler loaded!
Model: <class 'sklearn.svm._classes.SVC'>
Scaler: <class 'sklearn.preprocessing._data.StandardScaler'>


# 1. Load Trained Model

Load the RBF-SVM classifier and fitted `StandardScaler` saved by `nort.ipynb`. The scaler must be the same one used during training — applying it here ensures feature distributions match what the model was trained on.

In [2]:
def load_dlc_csv(dlc_path):
    df = pd.read_csv(dlc_path, header=[0, 1, 2], index_col=0)
    scorer = df.columns.get_level_values('scorer')[0]
    
    nose_x = df[scorer]['nose']['x'].values
    nose_y = df[scorer]['nose']['y'].values
    nose_likelihood = df[scorer]['nose']['likelihood'].values
    
    tail_x = df[scorer]['tail_base']['x'].values
    tail_y = df[scorer]['tail_base']['y'].values
    tail_likelihood = df[scorer]['tail_base']['likelihood'].values
    
    return {
        'nose_x': nose_x,
        'nose_y': nose_y,
        'nose_likelihood': nose_likelihood,
        'tail_x': tail_x,
        'tail_y': tail_y,
        'tail_likelihood': tail_likelihood
    }

def find_dlc_csv(video_path, dlc_folder):
    video_stem = Path(video_path).stem
    pattern = f"{video_stem}DLC*.csv"
    matches = [f for f in Path(dlc_folder).glob(pattern) if '_filtered' not in f.name]
    
    if len(matches) == 0:
        raise FileNotFoundError(f"No DLC CSV found for {video_stem}")
    
    return str(matches[0])

def bbox_to_vertices(bbox):
    xmin, ymin = bbox['xmin'], bbox['ymin']
    xmax, ymax = bbox['xmax'], bbox['ymax']
    
    vertices = np.array([
        [xmin, ymin],
        [xmax, ymin],
        [xmax, ymax],
        [xmin, ymax]
    ])
    return vertices

def calculate_features_for_frame(nose_x, nose_y, tail_x, tail_y, object_vertices):
    if np.isnan(nose_x) or np.isnan(nose_y) or np.isnan(tail_x) or np.isnan(tail_y):
        return np.full(8, np.nan)
    
    orientation = np.array([nose_x - tail_x, nose_y - tail_y])
    orientation_norm = np.linalg.norm(orientation)
    
    if orientation_norm < 1e-6:
        return np.full(8, np.nan)
    
    orientation = orientation / orientation_norm
    
    rotation_matrix = np.array([
        [orientation[0], orientation[1]],
        [-orientation[1], orientation[0]]
    ])
    
    features = []
    for vertex in object_vertices:
        vector = vertex - np.array([nose_x, nose_y])
        rotated = rotation_matrix @ vector
        features.extend(rotated)
    
    return np.array(features)

print("Helper functions loaded!")

Helper functions loaded!


# 2. Helper Functions

Utilities for loading DLC pose data and computing per-frame geometric features — identical to the feature engineering used during training in `nort.ipynb`.

- `load_dlc_csv` — parse DLC multi-header CSV into nose/tail arrays
- `find_dlc_csv` — locate the DLC CSV for a given video filename (handles naming variation)
- `bbox_to_vertices` — convert a `{xmin, ymin, xmax, ymax}` dict to a 4×2 vertex array
- `calculate_features_for_frame` — rotate bbox vertices into the mouse's head-direction frame and return the 8-feature vector for one object

In [3]:
def predict_with_object_assignment(video_path, svm_model, scaler, dlc_folder, bbox_assignments_dir):
    """Predict exploratory behavior per frame, then assign to closest object"""
    
    video_stem = Path(video_path).stem
    
    # Load bbox
    bbox_file = os.path.join(bbox_assignments_dir, f"{video_stem}_bbox_assignments.pkl")
    if not os.path.exists(bbox_file):
        print(f"Missing bbox for {video_stem}")
        return None
    
    with open(bbox_file, 'rb') as f:
        bbox_data = pickle.load(f)
    
    # Load DLC
    try:
        dlc_csv = find_dlc_csv(video_path, dlc_folder)
        dlc_data = load_dlc_csv(dlc_csv)
    except Exception as e:
        print(f"Error loading DLC for {video_stem}: {e}")
        return None
    
    n_frames = len(dlc_data['nose_x'])
    
    # Get object names
    first_frame_bboxes = bbox_data[0]
    object_names = sorted(first_frame_bboxes.keys())
    
    # Results
    frame_results = []
    
    for frame_idx in range(n_frames):
        if frame_idx not in bbox_data:
            frame_results.append({
                'frame_idx': frame_idx,
                'exploratory': False,
                'probability': 0.0,
                'exploring_object': None,
                'distance_to_objects': {}
            })
            continue
        
        bboxes = bbox_data[frame_idx]
        
        nose_x = dlc_data['nose_x'][frame_idx]
        nose_y = dlc_data['nose_y'][frame_idx]
        tail_x = dlc_data['tail_x'][frame_idx]
        tail_y = dlc_data['tail_y'][frame_idx]
        
        # Calculate features for ALL objects (16 features)
        all_features = []
        for obj_name in sorted(bboxes.keys()):
            vertices = bbox_to_vertices(bboxes[obj_name])
            features = calculate_features_for_frame(nose_x, nose_y, tail_x, tail_y, vertices)
            all_features.extend(features)
        
        all_features = np.array(all_features).reshape(1, -1)
        
        if np.any(np.isnan(all_features)):
            frame_results.append({
                'frame_idx': frame_idx,
                'exploratory': False,
                'probability': 0.0,
                'exploring_object': None,
                'distance_to_objects': {}
            })
            continue
        
        # Predict exploratory behavior for this frame
        features_scaled = scaler.transform(all_features)
        pred = svm_model.predict(features_scaled)[0]
        prob = svm_model.predict_proba(features_scaled)[0, 1]
        
        # Calculate distance to each object
        distances = {}
        for obj_name, bbox in bboxes.items():
            obj_center_x = (bbox['xmin'] + bbox['xmax']) / 2
            obj_center_y = (bbox['ymin'] + bbox['ymax']) / 2
            dist = np.sqrt((nose_x - obj_center_x)**2 + (nose_y - obj_center_y)**2)
            distances[obj_name] = dist
        
        # If exploratory, assign to closest object
        exploring_object = None
        if pred == 1:
            exploring_object = min(distances, key=distances.get)
        
        frame_results.append({
            'frame_idx': frame_idx,
            'exploratory': bool(pred),
            'probability': float(prob),
            'exploring_object': exploring_object,
            'distance_to_objects': distances
        })
    
    return {
        'video_stem': video_stem,
        'n_frames': n_frames,
        'object_names': object_names,
        'frame_results': frame_results
    }

def create_labeled_video(video_path, frame_results, output_path):
    """Create video with exploratory labels overlaid"""
    
    cap = cv2.VideoCapture(video_path)
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Create video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_idx < len(frame_results):
            result = frame_results[frame_idx]
            
            # Determine label and color
            if result['exploratory']:
                label = f"Exploring: {result['exploring_object']}"
                color = (0, 255, 0)  # Green
            else:
                label = "Non-exploratory"
                color = (0, 0, 255)  # Red
            
            # Add text to top right
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.7
            thickness = 2
            
            # Get text size for background
            (text_width, text_height), baseline = cv2.getTextSize(label, font, font_scale, thickness)
            
            # Position in top right
            x = width - text_width - 10
            y = 30
            
            # Draw background rectangle
            cv2.rectangle(frame, (x - 5, y - text_height - 5), (x + text_width + 5, y + 5), (0, 0, 0), -1)
            
            # Draw text
            cv2.putText(frame, label, (x, y), font, font_scale, color, thickness)
        
        out.write(frame)
        frame_idx += 1
    
    cap.release()
    out.release()

print("Prediction and video functions loaded!")

Prediction and video functions loaded!


# 3. Inference Pipeline

**`predict_with_object_assignment`** — the core inference function. For each frame:
1. Load the per-frame bbox assignment and DLC coordinates
2. Compute the 16-dimensional whole-frame feature vector (both objects concatenated)
3. Predict exploratory / non-exploratory using the SVM
4. If exploratory, compute nose-to-centroid distance for each object and assign to the closest one

Returns a dict with per-frame predictions, probabilities, assigned objects, and distances.

**`create_labeled_video`** — writes a new MP4 with the prediction overlaid as text in the top-right corner (green = exploratory, red = non-exploratory).

In [4]:
dlc_folder = "/mnt/g/Shared drives/llorente-lab/Moseq/AnalyzedData/Revision-NORT/Revision-NORT-Viewable"
bbox_assignments_dir = "/mnt/g/Shared drives/llorente-lab/Moseq/AnalyzedData/Revision-NORT/bbox_assignments"

all_videos = [f for f in os.listdir(dlc_folder) if f.endswith('.mp4')]
novel_videos = [v for v in all_videos if 'NOVEL' in v.upper()]

# 4. Select NOVEL Videos

Filter the video directory to NOVEL-phase recordings only (filenames containing `'NOVEL'`). Only the NOVEL phase is scored — the FAMILIAR phase is used for habituation and is not relevant to the discrimination index.

In [9]:
def process_and_save_novel_videos(novel_videos, svm_model, scaler, dlc_folder, bbox_assignments_dir):
    """Process all NOVEL videos and save results"""
    
    results_dir = "/mnt/g/Shared drives/llorente-lab/Moseq/AnalyzedData/Revision-NORT/results_nort"
    os.makedirs(results_dir, exist_ok=True)
    
    for video_file in tqdm(novel_videos, desc="Processing NOVEL videos"):
        video_path = os.path.join(dlc_folder, video_file)
        video_stem = Path(video_file).stem
        
        # Create subfolder for this video
        video_results_dir = os.path.join(results_dir, video_stem)
        os.makedirs(video_results_dir, exist_ok=True)
        
        # Predict
        result = predict_with_object_assignment(video_path, svm_model, scaler, dlc_folder, bbox_assignments_dir)
        
        if result is None:
            print(f"Skipped {video_stem}")
            continue
        
        # Save JSON results
        json_output = os.path.join(video_results_dir, f"{video_stem}_predictions.json")
        with open(json_output, 'w') as f:
            json.dump(result, f, indent=2)
        
        # Create labeled video
        video_output = os.path.join(video_results_dir, f"{video_stem}_labeled.mp4")
        create_labeled_video(video_path, result['frame_results'], video_output)
        
        print(f"Completed {video_stem}")

print("Processing all NOVEL videos...")
process_and_save_novel_videos(novel_videos, svm_optimized, scaler, dlc_folder, bbox_assignments_dir)
print("\nAll NOVEL videos processed!")

Processing all NOVEL videos...


Processing NOVEL videos:   1%|▎                                      | 1/140 [00:22<52:59, 22.87s/it]

Completed VK_20251026_a_BSL_NOVEL_20251029133242_ir_viewable


Processing NOVEL videos:   1%|▌                                    | 2/140 [00:55<1:06:05, 28.74s/it]

Completed VK_20251026_e_BSL_NOVEL_20251029132956_ir_viewable


...

Missing bbox for VK_20251026_j_30DPI_NOVEL_20251202143029_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled
Skipped VK_20251026_j_30DPI_NOVEL_20251202143029_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled
Missing bbox for VK_20251026_f_BSL_NOVEL_20251029133551_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled
Skipped VK_20251026_f_BSL_NOVEL_20251029133551_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled
Missing bbox for VK_20251026_aj_BSL_NOVEL_20251029173310_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled
Skipped VK_20251026_aj_BSL_NOVEL_20251029173310_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled
Missing bbox for VK_20251026_i_30DPI_NOVEL_20251202142406_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled
Skipped VK_20251026_i_30DPI_NOVEL_20251202142406_ir_viewable

Processing NOVEL videos:  93%|██████████████████████████████████▎  | 130/140 [31:22<00:03,  3.23it/s]

Missing bbox for VK_20251026_n_30DPI_NOVEL_20251202142840_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled
Skipped VK_20251026_n_30DPI_NOVEL_20251202142840_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled
Missing bbox for VK_20251026_g_30DPI_NOVEL_20251202133508_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled
Skipped VK_20251026_g_30DPI_NOVEL_20251202133508_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled
Missing bbox for VK_20251026_w_30DPI_NOVEL_20251202152740_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled
Skipped VK_20251026_w_30DPI_NOVEL_20251202152740_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled
Missing bbox for VK_20251026_ae_30DPI_NOVEL_20251202162114_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled
Skipped VK_20251026_ae_30DPI_NOVEL_20251202162114_ir_

Processing NOVEL videos: 100%|█████████████████████████████████████| 140/140 [31:22<00:00, 13.45s/it]

Missing bbox for VK_20251026_t_30DPI_NOVEL_20251202153001_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled
Skipped VK_20251026_t_30DPI_NOVEL_20251202153001_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled
Missing bbox for VK_20251026_ad_BSL_NOVEL_20251029162625_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled
Skipped VK_20251026_ad_BSL_NOVEL_20251029162625_ir_viewableDLC_Resnet101_NORTDec17shuffle2_snapshot_best-90_filtered_p60_labeled

All NOVEL videos processed!


In [10]:
# List all bbox files
bbox_files = sorted([f for f in os.listdir(bbox_assignments_dir) if f.endswith('.pkl')])

print(f"Total bbox files: {len(bbox_files)}")
print(f"\nFirst 10 bbox files:")
for i, f in enumerate(bbox_files[:10]):
    print(f"  {i+1}. {f}")

# Check NOVEL videos that are missing bboxes
novel_video_stems = [Path(v).stem for v in novel_videos]

missing_bbox = []
has_bbox = []

for video_stem in novel_video_stems:
    expected_bbox = f"{video_stem}_bbox_assignments.pkl"
    if expected_bbox in bbox_files:
        has_bbox.append(video_stem)
    else:
        missing_bbox.append(video_stem)

print(f"\n{'='*80}")
print(f"NOVEL videos WITH bbox: {len(has_bbox)}")
print(f"NOVEL videos MISSING bbox: {len(missing_bbox)}")

if len(missing_bbox) > 0:
    print(f"\nFirst 5 missing:")
    for i, v in enumerate(missing_bbox[:5]):
        print(f"  {i+1}. {v}")

Total bbox files: 137

First 10 bbox files:
  1. VIK_20251026_ag_BSL_NOVEL_20251029171530_ir_viewable_bbox_assignments.pkl
  2. VK_20251025_ag_30DPI_FAMILIAR_20251202164224_ir_viewable_bbox_assignments.pkl
  3. VK_20251025_ai_30DPI_FAMILIAR_20251202165520_ir_viewable_bbox_assignments.pkl
  4. VK_20251026_a_30DPI_FAMILIAR_20251202125804_ir_viewable_bbox_assignments.pkl
  5. VK_20251026_a_30DPI_NOVEL_20251202132421_ir_viewable_bbox_assignments.pkl
  6. VK_20251026_a_BSL_FAMILIAR_20251029130558_ir_viewable_bbox_assignments.pkl
  7. VK_20251026_a_BSL_NOVEL_20251029133242_ir_viewable_bbox_assignments.pkl
  8. VK_20251026_aa_30DPI_NOVEL_20251202162319_ir_viewable_bbox_assignments.pkl
  9. VK_20251026_aa_BSL_FAMILIAR_20251029160855_ir_viewable_bbox_assignments.pkl
  10. VK_20251026_aa_BSL_NOVEL_20251029163328_ir_viewable_bbox_assignments.pkl

NOVEL videos WITH bbox: 70
NOVEL videos MISSING bbox: 70

First 5 missing:
  1. VK_20251026_v_30DPI_NOVEL_20251202152109_ir_viewableDLC_Resnet101_NORTDe

# 5. Debug: Bbox File Matching

Half the NOVEL videos appear to be missing bbox files when matched by exact filename stem. The issue is that some video filenames include a DLC suffix (e.g., `...DLC_Resnet101_NORTDec17shuffle2_..._labeled`) appended to the base video name. The bbox files were saved using only the base stem before the `DLC` suffix. The next cell fixes this by stripping the suffix before lookup.

In [11]:
# Get NOVEL videos that actually have bbox files
novel_videos_with_bbox = []

for video_file in novel_videos:
    video_stem = Path(video_file).stem
    
    # Try exact match first
    expected_bbox = f"{video_stem}_bbox_assignments.pkl"
    if expected_bbox in bbox_files:
        novel_videos_with_bbox.append(video_file)
    else:
        # Try stripping DLC suffix
        if 'DLC' in video_stem:
            base_stem = video_stem.split('DLC')[0]
            expected_bbox = f"{base_stem}_bbox_assignments.pkl"
            if expected_bbox in bbox_files:
                novel_videos_with_bbox.append(video_file)

print(f"NOVEL videos with bbox files: {len(novel_videos_with_bbox)} / {len(novel_videos)}")
print(f"\nProcessing only these {len(novel_videos_with_bbox)} videos...")

NOVEL videos with bbox files: 140 / 140

Processing only these 140 videos...


In [12]:
def predict_with_object_assignment_fixed(video_path, svm_model, scaler, dlc_folder, bbox_assignments_dir):
    """Predict exploratory behavior per frame, handling DLC suffix in video names"""
    
    video_stem = Path(video_path).stem
    
    # Handle DLC suffix in video name
    bbox_stem = video_stem.split('DLC')[0] if 'DLC' in video_stem else video_stem
    bbox_file = os.path.join(bbox_assignments_dir, f"{bbox_stem}_bbox_assignments.pkl")
    
    if not os.path.exists(bbox_file):
        print(f"Missing bbox for {video_stem}")
        return None
    
    with open(bbox_file, 'rb') as f:
        bbox_data = pickle.load(f)
    
    # Load DLC
    try:
        dlc_csv = find_dlc_csv(video_path, dlc_folder)
        dlc_data = load_dlc_csv(dlc_csv)
    except Exception as e:
        print(f"Error loading DLC for {video_stem}: {e}")
        return None
    
    n_frames = len(dlc_data['nose_x'])
    
    # Get object names
    first_frame_bboxes = bbox_data[0]
    object_names = sorted(first_frame_bboxes.keys())
    
    # Results
    frame_results = []
    
    for frame_idx in range(n_frames):
        if frame_idx not in bbox_data:
            frame_results.append({
                'frame_idx': frame_idx,
                'exploratory': False,
                'probability': 0.0,
                'exploring_object': None,
                'distance_to_objects': {}
            })
            continue
        
        bboxes = bbox_data[frame_idx]
        
        nose_x = dlc_data['nose_x'][frame_idx]
        nose_y = dlc_data['nose_y'][frame_idx]
        tail_x = dlc_data['tail_x'][frame_idx]
        tail_y = dlc_data['tail_y'][frame_idx]
        
        # Calculate features for ALL objects (16 features)
        all_features = []
        for obj_name in sorted(bboxes.keys()):
            vertices = bbox_to_vertices(bboxes[obj_name])
            features = calculate_features_for_frame(nose_x, nose_y, tail_x, tail_y, vertices)
            all_features.extend(features)
        
        all_features = np.array(all_features).reshape(1, -1)
        
        if np.any(np.isnan(all_features)):
            frame_results.append({
                'frame_idx': frame_idx,
                'exploratory': False,
                'probability': 0.0,
                'exploring_object': None,
                'distance_to_objects': {}
            })
            continue
        
        # Predict exploratory behavior for this frame
        features_scaled = scaler.transform(all_features)
        pred = svm_model.predict(features_scaled)[0]
        prob = svm_model.predict_proba(features_scaled)[0, 1]
        
        # Calculate distance to each object
        distances = {}
        for obj_name, bbox in bboxes.items():
            obj_center_x = (bbox['xmin'] + bbox['xmax']) / 2
            obj_center_y = (bbox['ymin'] + bbox['ymax']) / 2
            dist = np.sqrt((nose_x - obj_center_x)**2 + (nose_y - obj_center_y)**2)
            distances[obj_name] = dist
        
        # If exploratory, assign to closest object
        exploring_object = None
        if pred == 1:
            exploring_object = min(distances, key=distances.get)
        
        frame_results.append({
            'frame_idx': frame_idx,
            'exploratory': bool(pred),
            'probability': float(prob),
            'exploring_object': exploring_object,
            'distance_to_objects': distances
        })
    
    return {
        'video_stem': video_stem,
        'n_frames': n_frames,
        'object_names': object_names,
        'frame_results': frame_results
    }

print("Updated prediction function loaded!")

Updated prediction function loaded!


# 6. Fixed Inference Function and Final Run

`predict_with_object_assignment_fixed` is identical to the original function but strips the DLC suffix from the video stem before looking up the bbox file. This resolves the missing-bbox issue and allows all 140 NOVEL videos to be processed.

The final `process_and_save_novel_videos` call runs inference on all videos and writes:
- `<video_stem>_predictions.json` — frame-level predictions, probabilities, assigned object, distances
- `<video_stem>_labeled.mp4` — annotated video with overlaid exploratory/non-exploratory labels

In [ ]:
def process_and_save_novel_videos(novel_videos, svm_model, scaler, dlc_folder, bbox_assignments_dir):
    """Process all NOVEL videos and save results"""
    
    results_dir = "/mnt/g/Shared drives/llorente-lab/Moseq/AnalyzedData/Revision-NORT/results_nort"
    os.makedirs(results_dir, exist_ok=True)
    
    for video_file in tqdm(novel_videos, desc="Processing NOVEL videos"):
        video_path = os.path.join(dlc_folder, video_file)
        video_stem = Path(video_file).stem
        
        # Create subfolder for this video
        video_results_dir = os.path.join(results_dir, video_stem)
        os.makedirs(video_results_dir, exist_ok=True)
        
        # Predict
        result = predict_with_object_assignment_fixed(video_path, svm_model, scaler, dlc_folder, bbox_assignments_dir)
        
        if result is None:
            print(f"Skipped {video_stem}")
            continue
        
        # Save JSON results
        json_output = os.path.join(video_results_dir, f"{video_stem}_predictions.json")
        with open(json_output, 'w') as f:
            json.dump(result, f, indent=2)
        
        # Create labeled video
        video_output = os.path.join(video_results_dir, f"{video_stem}_labeled.mp4")
        create_labeled_video(video_path, result['frame_results'], video_output)
        
        print(f"Completed {video_stem}")

print("Processing all NOVEL videos...")
process_and_save_novel_videos(novel_videos, svm_optimized, scaler, dlc_folder, bbox_assignments_dir)
print("\nAll NOVEL videos processed!")